# Zero-Shot Cloud Classification with Qwen3.5-0.8B (Local · Google Colab · 11 Cloud Classes)

This notebook runs **locally on a Colab GPU** using a **two-stage pipeline**:

| Stage | Model | Role |
|-------|-------|------|
| 1 — Vision | `Salesforce/blip-image-captioning-base` | Converts each cloud image → text description |
| 2 — Classification | `Qwen/Qwen3.5-0.8B` | Classifies the text description into one of 11 cloud classes |

`Qwen3.5-0.8B` is a **text-only** language model with no vision encoder, so a lightweight captioner
bridges the gap between pixels and language. Both models fit comfortably on a T4 GPU (16 GB).  
Results are saved to **Google Drive**.

**Runtime:** Enable GPU in *Runtime → Change runtime type → T4 GPU* before running.

## 1. GPU check & dependency installation

In [ ]:
import subprocess, sys

# Verify a GPU is available
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "WARNING: No GPU found — switch runtime to GPU!")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.45.0",
    "accelerate>=0.30.0",
    "datasets",
    "scikit-learn",
    "seaborn",
    "matplotlib",
    "pillow",
    "tqdm",
    "pandas",
])
print("Dependencies installed.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive mounted at /content/drive")

## 2b. (One-time) Unzip dataset from Drive  ← run only once, then skip

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run this cell ONCE to extract the dataset zip from Drive into /content/.
# Skip it on subsequent runs — the files stay in /content/ for the session.
#
# Step 1 (on your Mac): zip the dataset folder
#   cd /Users/nachomorerabarrios/Documentos/TFM
#   zip -r CloudDataset_Fixed.zip CloudDataset_Fixed/
#
# Step 2: upload CloudDataset_Fixed.zip to your Google Drive root.
#
# Step 3: run this cell — it unzips into /content/ (fast NVMe, not Drive FUSE).
# ─────────────────────────────────────────────────────────────────────────────
import os, zipfile

ZIP_PATH    = "/content/drive/MyDrive/CloudDataset_Fixed.zip"
EXTRACT_DIR = "/content/"   # unzips to /content/CloudDataset_Fixed/

if not os.path.exists(ZIP_PATH):
    print(f"Zip not found at {ZIP_PATH}. Upload it to Drive first.")
else:
    print(f"Extracting {ZIP_PATH} ...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print(f"Done. Dataset is at {EXTRACT_DIR}CloudDataset_Fixed/")
    # Verify
    for entry in sorted(os.listdir(f"{EXTRACT_DIR}CloudDataset_Fixed")):
        print(f"  {entry}/")

## 3. Configuration

In [ ]:
import os, re, random, math
import numpy as np

# ── Model ──────────────────────────────────────────────────────────────────────
# Stage-1 captioner (vision encoder — converts image → text)
CAPTION_MODEL_ID = "Salesforce/blip-image-captioning-base"
# Stage-2 classifier (text-only LLM — classifies the caption)
MODEL_ID = "Qwen/Qwen3.5-0.8B"

# ── Paths ─────────────────────────────────────────────────────────────────────
# DATASET_PATH points to /content/ (local Colab NVMe) after running cell 2b.
# Reading images from Drive FUSE is ~10-50× slower and prone to sync gaps.
DATASET_PATH = "/content/CloudDataset_Fixed"
OUTPUT_DIR   = "/content/drive/MyDrive/qwen35-08b-zeroshot-11cloudclasses"
TAG          = "zeroshot_qwen35_08b_11cloudclasses"

# ── Cloud classes (11-class taxonomy) ─────────────────────────────────────────
CLOUD_CLASSES = [
    "Altocumulus",
    "Altostratus",
    "Cirrocumulus",
    "Cirrus",
    "Clear Sky",
    "Contrails",
    "Cumulonimbus",
    "Cumulus",
    "Stratocumulus",
    "Stratus",
    "Veil",
]

NUM_TEST_IMAGES = None   # None → full test set; set an int to limit
SEED = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Model     : {MODEL_ID}")
print(f"Dataset   : {DATASET_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Classes ({len(CLOUD_CLASSES)}): {CLOUD_CLASSES}")
print(f"Num test images: {'all' if NUM_TEST_IMAGES is None else NUM_TEST_IMAGES}")

## 4. Verify dataset path

In [ ]:
assert os.path.isdir(DATASET_PATH), (
    f"Dataset not found at {DATASET_PATH}.\n"
    "Make sure the folder is uploaded to Google Drive and the path is correct."
)
print(f"Dataset found: {DATASET_PATH}")
for entry in sorted(os.listdir(DATASET_PATH)):
    full = os.path.join(DATASET_PATH, entry)
    suffix = "/" if os.path.isdir(full) else ""
    print(f"  - {entry}{suffix}")

## 5. Load dataset & filter to 11 standard cloud classes

In [ ]:
from PIL import Image
from collections import defaultdict

np.random.seed(SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}
RENAME_MAP = {"Veil Clouds": "Veil"}

def scan_split(split_dir):
    """
    Walk split_dir/ClassName/*.ext and return two parallel lists:
    image_paths and raw class-name strings.
    Ignores hidden files and non-image extensions.
    """
    paths, labels = [], []
    if not os.path.isdir(split_dir):
        return paths, labels
    for class_name in sorted(os.listdir(split_dir)):
        if class_name.startswith('.'):
            continue
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in sorted(os.listdir(class_dir)):
            if fname.startswith('.'):
                continue
            if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
                continue
            paths.append(os.path.join(class_dir, fname))
            labels.append(class_name)
    return paths, labels

# Locate split directories
VAL_CANDIDATES = ["val", "validation", "valid", "dev"]
top_entries    = sorted(os.listdir(DATASET_PATH))
split_dirs     = {e: os.path.join(DATASET_PATH, e)
                  for e in top_entries
                  if os.path.isdir(os.path.join(DATASET_PATH, e)) and not e.startswith('.')}
print(f"Folders in dataset: {list(split_dirs.keys())}")

train_dir = split_dirs.get("train")
test_dir  = split_dirs.get("test")
val_dir   = next((split_dirs[k] for k in VAL_CANDIDATES if k in split_dirs), None)

assert train_dir or test_dir, (
    f"No 'train' or 'test' folder found in {DATASET_PATH}.\n"
    f"Folders found: {list(split_dirs.keys())}"
)

# ── Deep-scan each split: print class folders AND sample file counts ─────
print("\nDetailed scan per split:")
for name, d in [("train", train_dir), ("test", test_dir), ("validation", val_dir)]:
    if not d:
        continue
    print(f"  [{name}] {d}")
    try:
        entries = [e for e in sorted(os.listdir(d)) if not e.startswith('.')]
    except Exception as exc:
        print(f"    ERROR listing dir: {exc}")
        continue
    for cls in entries[:20]:           # show up to 20 class folders
        cls_dir = os.path.join(d, cls)
        if not os.path.isdir(cls_dir):
            print(f"    {cls}  (file, skipped)")
            continue
        try:
            files = [f for f in os.listdir(cls_dir) if not f.startswith('.')]
            imgs  = [f for f in files if os.path.splitext(f)[1].lower() in IMAGE_EXTS]
            print(f"    {cls}/  total={len(files)}  images={len(imgs)}  "
                  f"sample={imgs[:3]}")
        except Exception as exc:
            print(f"    {cls}/  ERROR: {exc}")

# ── Scan all splits ───────────────────────────────────────────────────────
splits_raw = {}  # split_name -> (paths, raw_labels)
for name, d in [("train", train_dir), ("test", test_dir), ("validation", val_dir)]:
    if d:
        p, l = scan_split(d)
        print(f"  scan_split({name}): {len(p)} images, unique classes: {sorted(set(l))}")
        if p:
            splits_raw[name] = (p, l)

assert splits_raw, "No images found in any split! Check the paths printed above."

# Build original label list from train (or test as fallback)
ref_name        = "train" if "train" in splits_raw else next(iter(splits_raw))
original_labels = sorted(set(splits_raw[ref_name][1]))
print(f"\nOriginal classes ({len(original_labels)}): {original_labels}")

# Map original → new (rename + filter)
cloud_classes_set = set(CLOUD_CLASSES)
orig_to_new = {}
for lbl in original_labels:
    mapped = RENAME_MAP.get(lbl, lbl)
    orig_to_new[lbl] = mapped if mapped in cloud_classes_set else None
print(f"Label mapping: {orig_to_new}")

new_labels = CLOUD_CLASSES
label2id   = {l: i for i, l in enumerate(new_labels)}
NUM_LABELS = len(new_labels)
id2label   = {i: l for i, l in enumerate(new_labels)}

# Build lightweight per-split record lists: [{"path": ..., "label": int}, ...]
splits = {}  # split_name -> list of {"path", "label"}
for name, (paths, raw_labels) in splits_raw.items():
    records = []
    for path, raw in zip(paths, raw_labels):
        new = orig_to_new.get(raw)
        if new is not None:
            records.append({"path": path, "label": label2id[new]})
    splits[name] = records
    print(f"  {name} after filter: {len(records)} images")

print(f"\nCloud classes ({NUM_LABELS}): {new_labels}")

## 6. Select test set

In [ ]:
from collections import Counter

# Use 'test' if it exists, otherwise fall back to whatever split is available
TEST_SPLIT_NAME = "test" if "test" in splits else list(splits.keys())[-1]
print(f"Using split '{TEST_SPLIT_NAME}' as the evaluation set.")
test_records = splits[TEST_SPLIT_NAME]

if NUM_TEST_IMAGES is not None:
    idx = np.random.choice(len(test_records), size=NUM_TEST_IMAGES, replace=False)
    idx.sort()
    test_records = [test_records[i] for i in idx]

print(f"Test set: {len(test_records)} images")
label_counts = Counter(id2label[r["label"]] for r in test_records)
for label in new_labels:
    print(f"  {label}: {label_counts.get(label, 0)}")

## 7. Load model & processor

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BlipProcessor, BlipForConditionalGeneration,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")
    print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Stage-1: BLIP image captioner ────────────────────────────────────────────
print(f"\nLoading captioner: {CAPTION_MODEL_ID} ...")
caption_processor = BlipProcessor.from_pretrained(CAPTION_MODEL_ID)
caption_model = BlipForConditionalGeneration.from_pretrained(
    CAPTION_MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
caption_model.eval()
print("Captioner loaded.")

# ── Stage-2: Qwen3.5-0.8B text classifier ────────────────────────────────────
print(f"\nLoading classifier: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
model.eval()
print("Classifier loaded.")

## 8. Define classification prompt

In [ ]:
_labels_bullet = "\n".join(f"- {l}" for l in new_labels)

SYSTEM_PROMPT = (
    "You are a cloud classification expert with extensive knowledge of meteorology, "
    "atmospheric physics, and cloud types. "
    "You will be given a textual description of a sky/cloud image. "
    "Based on that description, classify it into exactly one of the following classes:\n\n"
    + _labels_bullet + "\n\n"
    "Justify your reasoning briefly, then place your final answer between "
    "<answer></answer> tags using the exact class name from the list above."
)

def make_user_prompt(caption: str) -> str:
    return (
        f"Image description: \"{caption}\"\n\n"
        "Based on this description, classify the cloud type."
    )

print(f"System prompt:\n{SYSTEM_PROMPT}")
print("\nExpected output format:")
print('  This looks like ... <answer>Cumulus</answer>')

## 9. Local inference function

In [ ]:
from PIL import Image

def caption_image(pil_image: Image.Image, max_new_tokens: int = 64) -> str:
    """Stage 1: generate a short text description of the image with BLIP."""
    img = pil_image.convert("RGB")
    inputs = caption_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = caption_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return caption_processor.decode(out[0], skip_special_tokens=True).strip()


def classify_caption(caption: str, max_new_tokens: int = 256) -> str:
    """Stage 2: classify a text caption with Qwen3.5-0.8B."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": make_user_prompt(caption)},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = out_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def classify_image_local(pil_image: Image.Image) -> str:
    """Full two-stage pipeline: image → caption → classification."""
    caption = caption_image(pil_image)
    return classify_caption(caption)


# Smoke test on the first image
print("Smoke test...")
_rec0    = test_records[0]
_img0    = Image.open(_rec0["path"])
_caption = caption_image(_img0)
print(f"True label : {id2label[_rec0['label']]}")
print(f"Caption    : {_caption}")
test_response = classify_caption(_caption)
print(f"Raw response:\n{test_response}")
print("\nPipeline ready.")

## 10. Output parser

In [ ]:
def _extract_label(text: str):
    """Find the first matching cloud class label in text."""
    for label in new_labels:
        if text.strip().lower() == label.lower():
            return label
    for label in sorted(new_labels, key=len, reverse=True):
        if label.lower() in text.lower():
            return label
    return None

def get_top1_prediction(raw: str) -> str:
    tag_match = re.search(r"<answer>\s*(.+?)\s*</answer>", raw, re.IGNORECASE | re.DOTALL)
    if tag_match:
        label = _extract_label(tag_match.group(1))
        if label:
            return label
    label = _extract_label(raw)
    if label:
        return label
    return "UNPARSEABLE"

def get_full_ranking(raw: str) -> list:
    pred = get_top1_prediction(raw)
    return [pred] if pred != "UNPARSEABLE" else []

def format_rate(raw: str) -> bool:
    return bool(re.search(r"<answer>\s*(.+?)\s*</answer>", raw, re.IGNORECASE))

print("Classification parser ready (11-class direct match).")
print(f"Classes: {new_labels}")

## 11. Run inference on test set

In [ ]:
import json
import time
from tqdm.auto import tqdm

RESUME = False   # set True to continue from a previous partial run

checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint_responses.jsonl")
done_indices = {}

if RESUME and os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            done_indices[entry["index"]] = entry
    print(f"Resuming: {len(done_indices)} already done out of {len(test_records)}")
elif os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Removed stale checkpoint, starting fresh.")

pending = [i for i in range(len(test_records)) if i not in done_indices]
print(f"Processing {len(pending)} images (sequential, GPU)...")

with open(checkpoint_path, "a") as ckpt_file:
    for i in tqdm(pending, desc="Classifying"):
        rec        = test_records[i]
        true_label = id2label[rec["label"]]
        t0         = time.time()
        raw        = classify_image_local(Image.open(rec["path"]))
        elapsed    = time.time() - t0
        pred       = get_top1_prediction(raw)
        ranking    = get_full_ranking(raw)

        result = {
            "index":        i,
            "true_label":   true_label,
            "pred_label":   pred,
            "ranking":      ranking,
            "raw_response": raw,
            "elapsed":      elapsed,
        }
        done_indices[i] = result
        ckpt_file.write(json.dumps(
            {k: result[k] for k in ("index", "true_label", "pred_label", "ranking", "raw_response")}
        ) + "\n")
        ckpt_file.flush()

ordered          = [done_indices[i] for i in range(len(test_records)) if i in done_indices]
true_labels_list = [r["true_label"]   for r in ordered]
pred_labels_list = [r["pred_label"]   for r in ordered]
rankings         = [r["ranking"]      for r in ordered]
raw_responses    = [r["raw_response"] for r in ordered]

accuracy_top1 = sum(t == p for t, p in zip(true_labels_list, pred_labels_list)) / len(true_labels_list)
fmt_rate      = sum(format_rate(r) for r in raw_responses) / len(raw_responses)
print(f"\nDone: {len(ordered)}/{len(test_records)} images")
print(f"Top-1 accuracy : {accuracy_top1:.4f}")
print(f"Format rate    : {fmt_rate:.4f}")

## 12. Classification report, confusion matrix & ranking metrics

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
)

top2 = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:2])  / len(true_labels_list)
top3 = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:3])  / len(true_labels_list)
top5 = sum(1 for t, r in zip(true_labels_list, rankings) if t in r[:5])  / len(true_labels_list)

mrr = np.mean([
    1.0 / (r.index(t) + 1) if t in r else 0.0
    for t, r in zip(true_labels_list, rankings)
])

def ndcg_at_k(true_label, ranking, k=NUM_LABELS):
    if true_label not in ranking:
        return 0.0
    pos = ranking.index(true_label)
    if pos >= k:
        return 0.0
    return 1.0 / math.log2(pos + 2)

ndcg = np.mean([ndcg_at_k(t, r) for t, r in zip(true_labels_list, rankings)])

valid_mask = [p in new_labels for p in pred_labels_list]
true_valid = [t for t, v in zip(true_labels_list, valid_mask) if v]
pred_valid = [p for p, v in zip(pred_labels_list, valid_mask) if v]
bal_acc    = balanced_accuracy_score(true_valid, pred_valid)
kappa      = cohen_kappa_score(true_valid, pred_valid, labels=new_labels)

print("=" * 60)
print(f"  Top-1  : {accuracy_top1:.4f}   Top-2  : {top2:.4f}")
print(f"  Top-3  : {top3:.4f}   Top-5  : {top5:.4f}")
print(f"  MRR    : {mrr:.4f}   NDCG@11: {ndcg:.4f}")
print(f"  Bal Acc: {bal_acc:.4f}   Cohen κ: {kappa:.4f}")
print(f"  Format : {fmt_rate:.4f}")
print("=" * 60)

report = classification_report(true_valid, pred_valid, labels=new_labels, digits=3, zero_division=0)
print(f"\nClassification Report\n{report}")

cm = confusion_matrix(true_valid, pred_valid, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted (Top-1)")
ax.set_ylabel("True")
ax.set_title(
    f"Confusion Matrix | Qwen3.5-0.8B Zero-Shot (11 Cloud Classes) | "
    f"Top-1={accuracy_top1:.3f} | MRR={mrr:.3f} | Bal Acc={bal_acc:.3f}"
)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{TAG}.png")
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved: {cm_path}")

## 13. Per-class accuracy, mean rank & nDCG@11

In [ ]:
per_class_top1      = {}
per_class_mean_rank = {}
per_class_ndcg      = {}

for label in new_labels:
    idxs = [i for i, t in enumerate(true_labels_list) if t == label]
    if not idxs:
        per_class_top1[label] = per_class_mean_rank[label] = per_class_ndcg[label] = float("nan")
        continue
    per_class_top1[label]      = sum(1 for i in idxs if pred_labels_list[i] == label) / len(idxs)
    per_class_mean_rank[label] = np.mean(
        [rankings[i].index(label) + 1 if label in rankings[i] else NUM_LABELS + 1 for i in idxs]
    )
    per_class_ndcg[label]      = np.mean([ndcg_at_k(label, rankings[i]) for i in idxs])

print(f"  {'Class':30s}  {'Top-1':>7s}  {'MeanRank':>9s}  {'nDCG@11':>9s}")
print("─" * 62)
for label in new_labels:
    print(
        f"  {label:30s}  {per_class_top1[label]:>7.3f}  "
        f"{per_class_mean_rank[label]:>9.2f}  "
        f"{per_class_ndcg[label]:>9.4f}"
    )
print("─" * 62)
print(
    f"  {'MACRO AVG':30s}  {accuracy_top1:>7.3f}  "
    f"{np.nanmean(list(per_class_mean_rank.values())):>9.2f}  "
    f"{np.nanmean(list(per_class_ndcg.values())):>9.4f}"
)

labels_sorted = sorted(new_labels, key=lambda l: per_class_top1.get(l, 0), reverse=True)
vals_sorted   = [per_class_top1.get(l, 0) for l in labels_sorted]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels_sorted, vals_sorted, color="#2196F3", edgecolor="black", linewidth=0.5)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Top-1 Accuracy")
ax.set_title(
    f"Per-Class Top-1 Accuracy — Qwen3.5-0.8B Zero-Shot | 11 Cloud Classes ({len(true_labels_list)} images)"
)
plt.xticks(rotation=45, ha="right")
for bar, val in zip(bars, vals_sorted):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{val:.2f}", ha="center", va="bottom", fontsize=8
    )
plt.tight_layout()
acc_path = os.path.join(OUTPUT_DIR, f"per_class_accuracy_{TAG}.png")
plt.savefig(acc_path, dpi=150)
plt.show()
print(f"Saved: {acc_path}")

## 14. Save results, metrics & final summary

In [ ]:
import json as _json

positions  = [r.index(t) + 1 if t in r else NUM_LABELS + 1 for t, r in zip(true_labels_list, rankings)]

results_df = pd.DataFrame({
    "true_label":   true_labels_list,
    "pred_label":   pred_labels_list,
    "correct":      [t == p for t, p in zip(true_labels_list, pred_labels_list)],
    "true_rank":    positions,
    "ranking":      ["|".join(r) for r in rankings],
    "raw_response": raw_responses,
})
csv_path = os.path.join(OUTPUT_DIR, f"predictions_{TAG}.csv")
results_df.to_csv(csv_path, index=False)
print(f"Saved predictions: {csv_path}")

metrics = {
    "model":        f"{CAPTION_MODEL_ID} + {MODEL_ID}",
    "n_images":     len(true_labels_list),
    "n_classes":    NUM_LABELS,
    "taxonomy":     "11 standard cloud classes",
    "classes":      new_labels,
    "top1":         round(accuracy_top1, 6),
    "top2":         round(top2, 6),
    "top3":         round(top3, 6),
    "top5":         round(top5, 6),
    "mrr":          round(float(mrr), 6),
    "ndcg":         round(float(ndcg), 6),
    "balanced_acc": round(float(bal_acc), 6),
    "cohen_kappa":  round(float(kappa), 6),
    "format_rate":  round(fmt_rate, 6),
}
metrics_path = os.path.join(OUTPUT_DIR, f"metrics_{TAG}.json")
with open(metrics_path, "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"Saved metrics:     {metrics_path}")

print()
print("=" * 65)
print("  FINAL RESULTS — Qwen3.5-0.8B Zero-Shot (11 Standard Cloud Classes)")
print("=" * 65)
print(f"  Model  : {MODEL_ID}")
print(f"  Classes: {new_labels}")
print(f"  Images : {len(true_labels_list)}")
print()
print(f"  Top-1: {accuracy_top1:.4f} | Top-2: {top2:.4f} | Top-3: {top3:.4f} | Top-5: {top5:.4f}")
print(f"  MRR: {mrr:.4f} | NDCG@11: {ndcg:.4f} | Bal Acc: {bal_acc:.4f} | Cohen κ: {kappa:.4f}")
print(f"  Format rate: {fmt_rate:.4f}")
print("=" * 65)